# Experiment 3 — Effect of factorial sparsity with optimized SAE hyperparameters

This notebook runs one experiment only: vary `N`, the number of named features in
each of two factor sets, while holding the data geometry, training budget, SAE
architecture, and optimizer configuration fixed. For every sample, exactly one `x`
feature and one `y` feature are active:

\[
P(x_i)=P(y_j)=\frac{1}{N}, \qquad P(x_i,y_j)=\frac{1}{N^2}.
\]

The ground-truth per-example L0 remains 2. Increasing `N` makes each named primitive
rarer across the dataset. The sweep uses `N = 2, 4, ..., 256`, five SAE seeds, and
exactly 125,000,000 generated activations per SAE.

The only reported true-feature recovery outcomes are:

- one-to-one Hungarian recovery at cosine `>= 0.90`;
- Mean Max Cosine Similarity (MMCS).

You must enter the optimized learning rate and L1 penalty in the required-input cell.
The notebook stops with an error when either value is missing.


## Environment setup (local or Google Colab)

Run this cell first. Locally, it finds the existing checkout and does not clone, pull,
or install packages. In Google Colab it clones the repository into `/content/SAE`
when absent, safely fast-forwards a clean existing clone, and installs
`composed-features/requirements.txt`.

Google Drive is never mounted. Checkpoints, CSVs, JSON metadata, and PNG plots remain
inside the local checkout under `/content/SAE/composed-features/artifacts/...` in
Colab. Because `/content` is runtime-local, commit, push, or download results before
resetting the runtime.


In [ ]:
from __future__ import annotations

import importlib.util
import os
import subprocess
import sys
from pathlib import Path


try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False

INSTALL_REQUIREMENTS = IN_COLAB
UPDATE_EXISTING_COLAB_REPOSITORY = True
COLAB_REPOSITORY_URL = 'https://github.com/Adefioye/SAE.git'
COLAB_REPOSITORY_ROOT = Path('/content/SAE')


def find_experiment_dir(start: Path) -> Path | None:
    candidates = (start, start / 'composed-features')
    return next(
        (
            path for path in candidates
            if (path / 'sparse_factorial_generator.py').is_file()
        ),
        None,
    )


if IN_COLAB:
    repository_root = COLAB_REPOSITORY_ROOT
    EXPERIMENT_DIR = find_experiment_dir(repository_root)
    if EXPERIMENT_DIR is None:
        if repository_root.exists() and any(repository_root.iterdir()):
            raise FileExistsError(
                f'{repository_root} exists but is not the SAE repository. '
                'Move that directory or choose a different clone path.'
            )
        repository_root.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            [
                'git', 'clone', '--depth', '1',
                COLAB_REPOSITORY_URL, str(repository_root),
            ],
            check=True,
        )
    elif UPDATE_EXISTING_COLAB_REPOSITORY:
        git_directory = repository_root / '.git'
        if git_directory.is_dir():
            repository_status = subprocess.run(
                ['git', '-C', str(repository_root), 'status', '--porcelain'],
                check=True,
                capture_output=True,
                text=True,
            ).stdout.strip()
            if repository_status:
                print('Existing Colab clone has local changes; skipping pull.')
            else:
                subprocess.run(
                    ['git', '-C', str(repository_root), 'pull', '--ff-only'],
                    check=True,
                )
        else:
            print('Existing project is not a Git clone; skipping pull.')
    EXPERIMENT_DIR = find_experiment_dir(repository_root)
else:
    EXPERIMENT_DIR = find_experiment_dir(Path.cwd().resolve())

if EXPERIMENT_DIR is None:
    raise FileNotFoundError(
        'Could not find composed-features/sparse_factorial_generator.py. '
        'Run locally from the repository root/composed-features directory, '
        'or verify the Colab repository URL.'
    )

EXPERIMENT_DIR = EXPERIMENT_DIR.resolve()
REQUIREMENTS_PATH = EXPERIMENT_DIR / 'requirements.txt'
if not REQUIREMENTS_PATH.is_file():
    raise FileNotFoundError(f'Missing requirements file: {REQUIREMENTS_PATH}')

if INSTALL_REQUIREMENTS:
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '--upgrade-strategy',
        'only-if-needed',
        '-r',
        str(REQUIREMENTS_PATH),
    ])

os.chdir(EXPERIMENT_DIR)
print('Runtime:', 'Google Colab' if IN_COLAB else 'local Jupyter')
print('Experiment directory:', EXPERIMENT_DIR)
print('Requirements installed:', INSTALL_REQUIREMENTS)
if IN_COLAB:
    print('Storage: /content runtime-local; Google Drive is not used.')


In [ ]:
import json
import math
import platform
import sys
from collections.abc import Iterator, Sequence
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from scipy.optimize import linear_sum_assignment
from scipy.stats import t as student_t
from tqdm.auto import tqdm


if 'EXPERIMENT_DIR' not in globals():
    working_dir = Path.cwd().resolve()
    if (working_dir / 'sparse_factorial_generator.py').is_file():
        EXPERIMENT_DIR = working_dir
    elif (working_dir / 'composed-features' / 'sparse_factorial_generator.py').is_file():
        EXPERIMENT_DIR = working_dir / 'composed-features'
    else:
        raise FileNotFoundError('Run the Environment setup cell first.')
EXPERIMENT_DIR = Path(EXPERIMENT_DIR).resolve()

if str(EXPERIMENT_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR))

import sae_lens
from sae_lens import StandardTrainingSAE, StandardTrainingSAEConfig
from sae_lens.config import LoggingConfig, SAETrainerConfig
from sae_lens.training.sae_trainer import SAETrainer

from sparse_factorial_generator import FactorialConfig, SparseFactorialGenerator
from toy_model import default_device


DEVICE = default_device()
ARTIFACT_ROOT = EXPERIMENT_DIR / 'artifacts' / '03_sparse_factorial_antipodal'
BASE_CHECKPOINT_ROOT = ARTIFACT_ROOT / 'sae_checkpoints'
BASE_RESULTS_ROOT = ARTIFACT_ROOT / 'results'
for path in (BASE_CHECKPOINT_ROOT, BASE_RESULTS_ROOT):
    path.mkdir(parents=True, exist_ok=True)

print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('CUDA device:', torch.cuda.get_device_name(DEVICE))
elif DEVICE.type == 'mps':
    print('Apple Metal acceleration enabled.')
print('SAELens:', sae_lens.__version__)
print('PyTorch:', torch.__version__)


## Required optimized hyperparameters

Replace both `None` values below with the selected values from Experiment 6. These
are the only two SAE/trainer hyperparameters that may differ from that experiment.

The five-seed `N` sweep is the notebook's only training experiment and is enabled by
default. Set its run flag to `False` only when you want to load and plot an already
completed result.


In [ ]:
# REQUIRED: replace None with your selected values, for example 3e-4 and 0.1.
OPTIMAL_LEARNING_RATE: float | None = None
OPTIMAL_L1_PENALTY: float | None = None

RUN_FIVE_SEED_N_SWEEP = True


def require_positive_finite(name: str, value: float | None) -> float:
    if value is None:
        raise ValueError(
            f'{name} is required. Replace None in this cell with the '
            'optimized value from Experiment 6.'
        )
    resolved = float(value)
    if not math.isfinite(resolved) or resolved <= 0:
        raise ValueError(f'{name} must be a positive finite number; got {value!r}.')
    return resolved


LEARNING_RATE = require_positive_finite(
    'OPTIMAL_LEARNING_RATE', OPTIMAL_LEARNING_RATE
)
L1_COEFFICIENT = require_positive_finite(
    'OPTIMAL_L1_PENALTY', OPTIMAL_L1_PENALTY
)

print('Optimized learning rate:', LEARNING_RATE)
print('Optimized L1 penalty:', L1_COEFFICIENT)
print('Five-seed N sweep enabled:', RUN_FIVE_SEED_N_SWEEP)


## Fixed Experiment 6 configuration and new artifact namespace

Every SAE uses exactly 125,000,000 samples. A run consists of 7,629 full batches of
16,384 plus one final batch of 6,464, for 7,630 optimizer steps. The smaller final
batch prevents silent sample-budget overshoot.

All existing Experiment 3 results remain untouched. The chosen learning rate and L1
penalty are encoded in a new results directory and checkpoint directory, so another
optimized pair also gets its own namespace.


In [ ]:
N_VALUES = (2, 4, 8, 16, 32, 64, 128, 256)
FIVE_SAE_SEEDS = tuple(range(5))
DICTIONARY_SEED = 0
DATA_SEED = 0

TRAINING_SAMPLES = 125_000_000
TRAIN_BATCH_SIZE = 16_384

# Fixed exactly as in Experiment 6.
ADAM_BETA1 = 0.0
ADAM_BETA2 = 0.999


def slug_value(value: float) -> str:
    return format(value, '.12g').replace('-', 'm').replace('.', 'p')


RUN_NAMESPACE = (
    'optimized_hparams_125m_5_seed_n_sweep_generated_antipodal_v1'
    f'__lr-{slug_value(LEARNING_RATE)}'
    f'__l1-{slug_value(L1_COEFFICIENT)}'
)
CHECKPOINT_ROOT = BASE_CHECKPOINT_ROOT / RUN_NAMESPACE
RESULTS_ROOT = BASE_RESULTS_ROOT / RUN_NAMESPACE
for path in (CHECKPOINT_ROOT, RESULTS_ROOT):
    path.mkdir(parents=True, exist_ok=True)

PER_SEED_RESULTS_PATH = RESULTS_ROOT / 'per_seed_results.csv'
AGGREGATE_RESULTS_PATH = RESULTS_ROOT / 'aggregate_results.csv'
RUN_CONFIG_PATH = RESULTS_ROOT / 'run_config.json'
COMBINED_PLOT_PATH = RESULTS_ROOT / 'true_feature_recovery_by_n.png'
HUNGARIAN_PLOT_PATH = RESULTS_ROOT / 'hungarian_90_by_n.png'
MMCS_PLOT_PATH = RESULTS_ROOT / 'mmcs_by_n.png'

full_batches, final_batch_size = divmod(TRAINING_SAMPLES, TRAIN_BATCH_SIZE)
EXPECTED_OPTIMIZER_STEPS = full_batches + int(final_batch_size > 0)
assert len(FIVE_SAE_SEEDS) == 5

RUN_CONFIG = {
    'study': 'optimized_hparams_125m_5_seed_n_sweep_generated_antipodal_v1',
    'n_values': list(N_VALUES),
    'sae_seeds': list(FIVE_SAE_SEEDS),
    'dictionary_seed': DICTIONARY_SEED,
    'data_seed': DATA_SEED,
    'training_samples_per_sae': TRAINING_SAMPLES,
    'train_batch_size': TRAIN_BATCH_SIZE,
    'final_batch_size': final_batch_size,
    'optimizer_steps_per_sae': EXPECTED_OPTIMIZER_STEPS,
    'learning_rate': LEARNING_RATE,
    'l1_coefficient': L1_COEFFICIENT,
    'n_sae_runs': len(N_VALUES) * len(FIVE_SAE_SEEDS),
    'sae_lens_version': sae_lens.__version__,
    'torch_version': torch.__version__,
    'python_version': platform.python_version(),
}
RUN_CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2) + '\n')

print('N values:', N_VALUES)
print('SAE seeds:', FIVE_SAE_SEEDS)
print('SAE runs:', RUN_CONFIG['n_sae_runs'])
print('Samples per SAE:', f'{TRAINING_SAMPLES:,}')
print('Batch size:', f'{TRAIN_BATCH_SIZE:,}')
print('Final batch:', f'{final_batch_size:,}')
print('Optimizer steps per SAE:', f'{EXPECTED_OPTIMIZER_STEPS:,}')
print('Checkpoint root:', CHECKPOINT_ROOT)
print('Results root:', RESULTS_ROOT)


## Orthogonal-antipodal dictionaries

For every even `N`, including `N=2`, both factor sets consist of unit-norm
antipodal axis pairs in orthogonal subspaces, followed by a deterministic random
orthogonal rotation. This construction has no trained-model or checkpoint dependency.


In [ ]:
def orthogonal_matrix(dim: int, seed: int) -> torch.Tensor:
    generator = torch.Generator(device='cpu').manual_seed(seed)
    matrix = torch.randn(dim, dim, generator=generator)
    q, r = torch.linalg.qr(matrix)
    signs = torch.where(torch.diag(r) >= 0, 1.0, -1.0)
    return q * signs.unsqueeze(0)


def make_antipodal_dictionary(
    n_per_set: int,
    *,
    seed: int,
    device: torch.device | str,
) -> torch.Tensor:
    if n_per_set <= 0 or n_per_set % 2:
        raise ValueError('Antipodal geometry requires a positive, even N.')
    half = n_per_set // 2
    dictionary = torch.zeros(2 * n_per_set, n_per_set)
    for axis in range(half):
        dictionary[2 * axis, axis] = 1.0
        dictionary[2 * axis + 1, axis] = -1.0

        y_offset = n_per_set
        y_axis = half + axis
        dictionary[y_offset + 2 * axis, y_axis] = 1.0
        dictionary[y_offset + 2 * axis + 1, y_axis] = -1.0

    return (dictionary @ orthogonal_matrix(n_per_set, seed)).to(device)


def verify_antipodal_dictionary(
    dictionary: torch.Tensor, n_per_set: int
) -> None:
    unit = F.normalize(dictionary, dim=1)
    x_group = unit[:n_per_set]
    y_group = unit[n_per_set:]
    for group in (x_group, y_group):
        assert torch.allclose(group[0::2], -group[1::2], atol=2e-4)
    assert float((x_group @ y_group.T).abs().max()) < 2e-4


for n_per_set in N_VALUES:
    verify_antipodal_dictionary(
        make_antipodal_dictionary(
            n_per_set, seed=DICTIONARY_SEED, device=DEVICE
        ),
        n_per_set,
    )
print('Verified all orthogonal-antipodal dictionaries.')


## SAE construction and exact-budget training

Only `d_in` and `d_sae` scale with `N`; they are structural consequences of the
experiment rather than tuned hyperparameters. Apart from the required learning rate
and L1 penalty, the SAE and trainer settings below exactly match Experiment 6.


In [ ]:
@dataclass(frozen=True)
class ExperimentSpec:
    n_per_set: int
    sae_seed: int
    dictionary_seed: int = DICTIONARY_SEED
    data_seed: int = DATA_SEED
    training_samples: int = TRAINING_SAMPLES
    batch_size: int = TRAIN_BATCH_SIZE
    learning_rate: float = LEARNING_RATE
    l1_coefficient: float = L1_COEFFICIENT

    @property
    def activation_dim(self) -> int:
        return self.n_per_set

    @property
    def d_sae(self) -> int:
        return 2 * self.n_per_set

    def checkpoint_slug(self) -> str:
        return (
            f'N-{self.n_per_set:04d}'
            f'__dict-{self.dictionary_seed}'
            f'__data-{self.data_seed}'
            f'__sae-{self.sae_seed}'
            f'__T-{self.training_samples}'
            f'__batch-{self.batch_size}'
        )


def build_dictionary(spec: ExperimentSpec) -> torch.Tensor:
    return make_antipodal_dictionary(
        spec.n_per_set,
        seed=spec.dictionary_seed,
        device=DEVICE,
    )


def build_generator(spec: ExperimentSpec) -> SparseFactorialGenerator:
    return SparseFactorialGenerator(
        FactorialConfig(
            n_x=spec.n_per_set,
            n_y=spec.n_per_set,
            activation_dim=spec.activation_dim,
            amplitude_correlation=1.0,
            singleton_probability=0.0,
            noise_std=0.0,
            seed=spec.data_seed,
        ),
        device=str(DEVICE),
        true_dictionary=build_dictionary(spec),
        normalize_dictionary=False,
    )


class FixedBatchActivationIterator(Iterator[torch.Tensor]):
    def __init__(
        self, generator: SparseFactorialGenerator, batch_size: int
    ) -> None:
        self.generator = generator
        self.batch_size = batch_size

    def __iter__(self):
        return self

    @torch.no_grad()
    def __next__(self) -> torch.Tensor:
        hidden, _ = self.generator.sample(self.batch_size)
        return hidden.detach()


def make_standard_sae(spec: ExperimentSpec) -> StandardTrainingSAE:
    torch.manual_seed(spec.sae_seed)
    cfg = StandardTrainingSAEConfig(
        d_in=spec.activation_dim,
        d_sae=spec.d_sae,
        device=str(DEVICE),
        dtype='float32',
        apply_b_dec_to_input=False,
        normalize_activations='none',
        l1_coefficient=spec.l1_coefficient,
        lp_norm=1.0,
        l1_warm_up_steps=0,
        decoder_init_norm=1.0,
    )
    return StandardTrainingSAE(cfg).to(DEVICE)


def make_trainer(
    spec: ExperimentSpec,
    sae: StandardTrainingSAE,
    generator: SparseFactorialGenerator,
) -> SAETrainer:
    cfg = SAETrainerConfig(
        total_training_samples=spec.training_samples,
        train_batch_size_samples=spec.batch_size,
        lr=spec.learning_rate,
        lr_end=spec.learning_rate,
        lr_scheduler_name='constant',
        lr_warm_up_steps=0,
        lr_decay_steps=0,
        adam_beta1=ADAM_BETA1,
        adam_beta2=ADAM_BETA2,
        device=str(DEVICE),
        n_checkpoints=0,
        checkpoint_path=None,
        save_final_checkpoint=False,
        logger=LoggingConfig(
            log_to_wandb=False,
            eval_every_n_wandb_logs=2**31 - 1,
        ),
        n_batches_for_norm_estimate=1,
    )
    return SAETrainer(
        cfg=cfg,
        sae=sae,
        data_provider=FixedBatchActivationIterator(generator, spec.batch_size),
    )


def train_exact_sample_budget(
    trainer: SAETrainer,
    generator: SparseFactorialGenerator,
    *,
    description: str,
) -> StandardTrainingSAE:
    pbar = tqdm(
        total=TRAINING_SAMPLES,
        initial=trainer.n_training_samples,
        desc=description,
    )
    while trainer.n_training_samples < TRAINING_SAMPLES:
        remaining = TRAINING_SAMPLES - trainer.n_training_samples
        current_batch_size = min(TRAIN_BATCH_SIZE, remaining)
        hidden, _ = generator.sample(current_batch_size)
        trainer.maybe_reset_sparsity()
        step_output = trainer.step(hidden.detach())
        if not bool(torch.isfinite(step_output.loss).all()):
            pbar.close()
            raise FloatingPointError(
                f'Non-finite loss at optimizer step {trainer.n_training_steps:,}.'
            )
        trainer.n_training_steps += 1
        pbar.update(current_batch_size)

    if trainer.n_training_samples != TRAINING_SAMPLES:
        raise RuntimeError('Trainer did not stop at the exact sample budget.')
    trainer.set_final_sae_metadata()
    pbar.close()
    return trainer.sae


In [ ]:
def checkpoint_spec(spec: ExperimentSpec) -> dict:
    return {
        **asdict(spec),
        'activation_dim': spec.activation_dim,
        'd_sae': spec.d_sae,
        'optimizer_steps': EXPECTED_OPTIMIZER_STEPS,
        'apply_b_dec_to_input': False,
        'normalize_activations': 'none',
        'lp_norm': 1.0,
        'l1_warm_up_steps': 0,
        'decoder_init_norm': 1.0,
        'lr_scheduler_name': 'constant',
        'lr_warm_up_steps': 0,
        'lr_decay_steps': 0,
        'adam_beta1': ADAM_BETA1,
        'adam_beta2': ADAM_BETA2,
        'sae_lens_version': sae_lens.__version__,
    }


def checkpoint_matches(path: Path, expected: dict) -> bool:
    required = ('cfg.json', 'sae_weights.safetensors', 'experiment_spec.json')
    if not all((path / filename).is_file() for filename in required):
        return False
    try:
        saved = json.loads((path / 'experiment_spec.json').read_text())
    except (OSError, json.JSONDecodeError):
        return False
    return all(saved.get(key) == value for key, value in expected.items())


def train_or_load_sae(
    spec: ExperimentSpec,
) -> tuple[StandardTrainingSAE, str, int]:
    path = CHECKPOINT_ROOT / spec.checkpoint_slug()
    expected = checkpoint_spec(spec)
    if checkpoint_matches(path, expected):
        sae = StandardTrainingSAE.load_from_disk(
            path, device=str(DEVICE), dtype='float32'
        ).to(DEVICE)
        print('Loaded checkpoint:', path.name)
        return sae, 'loaded_checkpoint', EXPECTED_OPTIMIZER_STEPS

    if path.exists():
        raise ValueError(f'Checkpoint exists but does not match this run: {path}')

    generator = build_generator(spec)
    sae = make_standard_sae(spec)
    trainer = make_trainer(spec, sae, generator)
    sae = train_exact_sample_budget(
        trainer,
        generator,
        description=f'N={spec.n_per_set}, SAE seed={spec.sae_seed}',
    )
    path.mkdir(parents=True, exist_ok=False)
    sae.save_model(path)
    expected['device_used_to_train'] = str(DEVICE)
    (path / 'experiment_spec.json').write_text(
        json.dumps(expected, indent=2) + '\n'
    )
    print('Saved checkpoint:', path.name)
    return sae, 'new_training_run', trainer.n_training_steps


def clear_accelerator_cache() -> None:
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    elif DEVICE.type == 'mps':
        torch.mps.empty_cache()


## Two true-feature recovery metrics

Let `S` be the cosine-similarity matrix between all `2N` true primitives and all
`2N` learned decoder directions.

- **MMCS:** average, over true primitives, of the largest cosine in each row of `S`.
- **Hungarian 90%:** fraction of globally optimal one-to-one assignments whose cosine
  is at least `0.90`.

Higher is better for both. MMCS does not enforce unique decoder assignments, making
Hungarian 90% the stricter complementary measure.


In [ ]:
@torch.no_grad()
def primitive_recovery_metrics(
    sae: StandardTrainingSAE,
    dictionary: torch.Tensor,
) -> dict[str, float]:
    decoder = F.normalize(sae.W_dec.detach().float(), dim=1)
    primitives = F.normalize(dictionary.detach().float(), dim=1)
    similarity = primitives @ decoder.T

    mmcs = float(similarity.max(dim=1).values.mean())
    primitive_rows, decoder_cols = linear_sum_assignment(
        (-similarity).detach().cpu().numpy()
    )
    assigned = similarity[
        torch.as_tensor(primitive_rows, device=similarity.device),
        torch.as_tensor(decoder_cols, device=similarity.device),
    ]
    return {
        'hungarian_90': float((assigned >= 0.90).float().mean()),
        'mmcs': mmcs,
    }


## Run the five-seed `N` sweep

There are 40 independent SAE runs: eight `N` values times five SAE initialization
seeds. Dictionary and generated-data seeds remain fixed at zero. The per-seed CSV is
atomically updated after every completed SAE, and completed rows/checkpoints are
reused on rerun.


In [ ]:
RESULT_COLUMNS = [
    'n_per_set',
    'sae_seed',
    'dictionary_seed',
    'data_seed',
    'training_samples',
    'train_batch_size',
    'optimizer_steps',
    'learning_rate',
    'l1_coefficient',
    'checkpoint_source',
    'hungarian_90',
    'mmcs',
]


def save_results(frame: pd.DataFrame, path: Path) -> None:
    temporary_path = path.with_suffix('.tmp')
    frame[RESULT_COLUMNS].to_csv(temporary_path, index=False)
    temporary_path.replace(path)


def load_completed_results(path: Path) -> dict[tuple[int, int], dict]:
    if not path.is_file():
        return {}
    frame = pd.read_csv(path)
    missing = set(RESULT_COLUMNS).difference(frame.columns)
    if missing:
        raise ValueError(f'{path} is missing columns: {sorted(missing)}')

    invariants = {
        'training_samples': TRAINING_SAMPLES,
        'train_batch_size': TRAIN_BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'l1_coefficient': L1_COEFFICIENT,
        'dictionary_seed': DICTIONARY_SEED,
        'data_seed': DATA_SEED,
    }
    for column, expected in invariants.items():
        if not all(
            math.isclose(float(value), float(expected), rel_tol=1e-12, abs_tol=1e-15)
            for value in frame[column]
        ):
            raise ValueError(
                f'{path} contains {column} values that do not match {expected}.'
            )

    completed = {}
    for row in frame[RESULT_COLUMNS].to_dict(orient='records'):
        key = (int(row['n_per_set']), int(row['sae_seed']))
        if key in completed:
            raise ValueError(f'Duplicate saved result key: {key}')
        completed[key] = row
    return completed


SPECS = [
    ExperimentSpec(n_per_set=n_per_set, sae_seed=sae_seed)
    for n_per_set in N_VALUES
    for sae_seed in FIVE_SAE_SEEDS
]
EXPECTED_KEYS = [(spec.n_per_set, spec.sae_seed) for spec in SPECS]


def run_five_seed_n_sweep() -> pd.DataFrame:
    completed = load_completed_results(PER_SEED_RESULTS_PATH)
    unexpected = set(completed).difference(EXPECTED_KEYS)
    if unexpected:
        raise ValueError(f'Unexpected saved result keys: {sorted(unexpected)}')

    for index, spec in enumerate(SPECS, start=1):
        key = (spec.n_per_set, spec.sae_seed)
        if key in completed:
            print(
                f'[{index}/{len(SPECS)}] Reused CSV row: '
                f'N={spec.n_per_set}, SAE seed={spec.sae_seed}'
            )
            continue

        print(
            f'[{index}/{len(SPECS)}] Training N={spec.n_per_set}, '
            f'SAE seed={spec.sae_seed}'
        )
        sae, checkpoint_source, optimizer_steps = train_or_load_sae(spec)
        metrics = primitive_recovery_metrics(sae, build_dictionary(spec))
        completed[key] = {
            'n_per_set': spec.n_per_set,
            'sae_seed': spec.sae_seed,
            'dictionary_seed': spec.dictionary_seed,
            'data_seed': spec.data_seed,
            'training_samples': spec.training_samples,
            'train_batch_size': spec.batch_size,
            'optimizer_steps': optimizer_steps,
            'learning_rate': spec.learning_rate,
            'l1_coefficient': spec.l1_coefficient,
            'checkpoint_source': checkpoint_source,
            **metrics,
        }
        progress = pd.DataFrame(
            [completed[item] for item in EXPECTED_KEYS if item in completed]
        ).sort_values(['n_per_set', 'sae_seed']).reset_index(drop=True)
        save_results(progress, PER_SEED_RESULTS_PATH)
        print(
            f'Saved result: H90={metrics["hungarian_90"]:.4f}, '
            f'MMCS={metrics["mmcs"]:.6f}'
        )
        del sae
        clear_accelerator_cache()

    missing_runs = [key for key in EXPECTED_KEYS if key not in completed]
    if missing_runs:
        raise RuntimeError(f'Missing completed runs: {missing_runs}')
    return pd.DataFrame(
        [completed[key] for key in EXPECTED_KEYS]
    )[RESULT_COLUMNS].sort_values(
        ['n_per_set', 'sae_seed']
    ).reset_index(drop=True)


if RUN_FIVE_SEED_N_SWEEP:
    per_seed_results = run_five_seed_n_sweep()
elif PER_SEED_RESULTS_PATH.is_file():
    per_seed_results = pd.DataFrame(
        load_completed_results(PER_SEED_RESULTS_PATH).values()
    )[RESULT_COLUMNS].sort_values(
        ['n_per_set', 'sae_seed']
    ).reset_index(drop=True)
else:
    raise FileNotFoundError(
        f'No saved results at {PER_SEED_RESULTS_PATH}. '
        'Set RUN_FIVE_SEED_N_SWEEP=True to train the experiment.'
    )

display(per_seed_results)
print('Saved per-seed results:', PER_SEED_RESULTS_PATH)


## Aggregate five seeds and save recovery plots

For each `N`, the aggregate CSV contains the mean and two-sided 95% Student-t
confidence interval across the five SAE seeds. The combined PNG has one subplot per
recovery metric; individual metric PNGs are also saved.


In [ ]:
def aggregate_five_seed_recovery(frame: pd.DataFrame) -> pd.DataFrame:
    records = []
    expected_seeds = set(FIVE_SAE_SEEDS)
    for n_per_set, group in frame.groupby('n_per_set', sort=True):
        observed_seeds = set(group['sae_seed'].astype(int))
        if observed_seeds != expected_seeds:
            raise ValueError(
                f'N={int(n_per_set)} has seeds {sorted(observed_seeds)}; '
                f'expected {sorted(expected_seeds)}.'
            )

        record = {
            'n_per_set': int(n_per_set),
            'n_sae_seeds': len(group),
            'training_samples_per_sae': TRAINING_SAMPLES,
            'train_batch_size': TRAIN_BATCH_SIZE,
            'learning_rate': LEARNING_RATE,
            'l1_coefficient': L1_COEFFICIENT,
        }
        for metric in ('hungarian_90', 'mmcs'):
            values = group[metric].astype(float)
            mean = float(values.mean())
            standard_error = float(values.std(ddof=1) / math.sqrt(len(values)))
            half_width = float(
                student_t.ppf(0.975, df=len(values) - 1) * standard_error
            )
            record[f'{metric}_mean'] = mean
            record[f'{metric}_ci95_lower'] = mean - half_width
            record[f'{metric}_ci95_upper'] = mean + half_width
        records.append(record)
    return pd.DataFrame(records)


aggregate_results = aggregate_five_seed_recovery(per_seed_results)
aggregate_results.to_csv(AGGREGATE_RESULTS_PATH, index=False)
display(aggregate_results.style.format(precision=5))
print('Saved aggregate results:', AGGREGATE_RESULTS_PATH)


PLOT_SPECS = (
    ('hungarian_90', 'Hungarian alignment — 90% cutoff',
     'True-feature recovery fraction', HUNGARIAN_PLOT_PATH),
    ('mmcs', 'Mean Max Cosine Similarity', 'MMCS', MMCS_PLOT_PATH),
)


def plot_metric_with_ci(ax, frame, metric, title, ylabel):
    ordered = frame.sort_values('n_per_set')
    means = ordered[f'{metric}_mean']
    lower = means - ordered[f'{metric}_ci95_lower']
    upper = ordered[f'{metric}_ci95_upper'] - means
    ax.errorbar(
        ordered['n_per_set'],
        means,
        yerr=[lower, upper],
        marker='o',
        linewidth=2,
        capsize=4,
    )
    ax.set(
        title=title,
        xlabel='Number of features per set (N)',
        ylabel=ylabel,
        ylim=(-0.05, 1.05),
    )
    ax.set_xscale('log', base=2)
    ax.set_xticks(ordered['n_per_set'])
    ax.set_xticklabels([str(int(value)) for value in ordered['n_per_set']])
    ax.get_xaxis().set_minor_formatter(plt.NullFormatter())
    ax.grid(alpha=0.25)


figure, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
for axis, (metric, title, ylabel, individual_path) in zip(
    axes, PLOT_SPECS, strict=True
):
    plot_metric_with_ci(axis, aggregate_results, metric, title, ylabel)

    individual_figure, individual_axis = plt.subplots(figsize=(6, 4.5))
    plot_metric_with_ci(
        individual_axis, aggregate_results, metric, title, ylabel
    )
    individual_figure.tight_layout()
    individual_figure.savefig(individual_path, dpi=200, bbox_inches='tight')
    plt.close(individual_figure)
    print('Saved plot:', individual_path)

figure.savefig(COMBINED_PLOT_PATH, dpi=200, bbox_inches='tight')
print('Saved combined plot:', COMBINED_PLOT_PATH)
plt.show()


## Interpretation

- An upward trend with `N` supports the claim that decreasing named-primitive
  frequency encourages recovery of the true primitive dictionary.
- High MMCS with lower Hungarian 90% means primitives have nearby learned directions
  but do not consistently receive distinct decoder slots.
- The confidence intervals summarize SAE initialization/training variability across
  seeds 0–4; dictionary and generated-data randomness are intentionally fixed.

All new outputs are isolated under the optimized-hyperparameter namespace printed by
the configuration cell. Existing 250M and 128M Experiment 3 artifacts are not read,
overwritten, or treated as checkpoints for this experiment.
